# 🧨 Hotel Reservation System 4: Edge Cases and Errors

The core system from `03_hotel_compliance_checker.ipynb` works on friendly requests. This notebook pushes on its edges: a request with
SQL injection in it, a policy violation that should stop for a human, and booking dates that are already in the
past. Each edge gets a safeguard, and each safeguard is checked in code.

## Learning Objectives
1. **Injection-style input** - What happens when the guest's text ends up inside SQL the agent writes
2. **Human review with `interrupt()`** - Pause the graph on a `VIOLATION`, then resume with `Command(resume=...)`
3. **Inspecting a thread** - `get_state()` and `get_state_history()` show where a run stopped and how it got there
4. **A past-date guard** - Structured output extracts the dates; plain code compares them with today

## Where this fits
`03_hotel_compliance_checker.ipynb` built the core system: coordinator, compliance checker and reservation
agent. This notebook imports that core from `hotel_common` and hardens it. Nodes that fail now and then
(`RetryPolicy`) and a log file of failures are in `09_hotel_system_wide_error_handling.ipynb`.
`05_hotel_testing_the_system.ipynb` then turns these checks into a small test suite.

## Prerequisites
- The core system from `03_hotel_compliance_checker.ipynb` (read it first; this notebook imports it from `hotel_common.py`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import sqlite3
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by all agents and the date check
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧰 Part 1: Data and Tools

The database, 02's SQL tools (`run_query` only reads, `book_room` is the only write) and 03's compliance
retriever. `count_rows()` reads table sizes straight from SQLite, so the checks below never have to trust a reply.

In [ ]:
# ============================================================================
# DATABASE AND TOOLS
# ============================================================================
db_path = hc.init_hotel_db("hotel.db")
sql_tools = hc.make_sql_tools(db_path)  # list_tables, describe_table, run_query (read-only) and book_room
SQL_TOOL_NAMES = {t.name for t in sql_tools}
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]


def count_rows(table: str) -> int:
    with sqlite3.connect("hotel.db") as conn:
        return conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]


print(f"✅ {db_path}: {count_rows('rooms')} rooms, {count_rows('reservations')} reservations")
print(f"✅ SQL tools {sorted(SQL_TOOL_NAMES)}, 1 retrieval tool")

---
## 🏗️ Part 2: The Core System, as a Builder Function

The reservation agent, the compliance checker and the router are 03's, imported from `hotel_common`. One node is
upgraded and written out here: `generate` now gets its verdict from `with_structured_output(ComplianceVerdict)`,
so it is always exactly `COMPLIANT` or `VIOLATION` instead of a word to search for in free text. That matters
here, because a `VIOLATION` will stop the graph for a human. (`hotel_common` keeps this version for the later
notebooks: `hc.make_core_nodes()` and `hc.latest_verdict()`.)

The demos below need variants of the same graph (with a review step, with a date check). `build_graph()` takes
the parts that change as arguments, so each variant is one line.

In [ ]:
# ============================================================================
# CORE NODES, ROUTERS AND THE GRAPH BUILDER
# ============================================================================
AGENTS = ["reservation_assistant", "compliance_checker"]
conversation_prompt = hc.conversation_prompt(AGENTS, extra_guidelines=hc.COMPLIANCE_FIRST)  # 03's guideline
sql_assistant, rag_assistant, _ = hc.make_core_nodes(llm, sql_tools, rag_tools)  # 03's agents; generate is below


def conv_assistant(state: MessagesState) -> dict:
    return {"messages": [llm.invoke([conversation_prompt] + state["messages"])]}


class ComplianceVerdict(BaseModel):
    verdict: Literal["COMPLIANT", "VIOLATION"] = Field(description="VIOLATION only if a retrieved rule is broken")
    rule: int | None = Field(description="Number of the violated rule, or null if none")
    reason: str = Field(description="One sentence that cites the retrieved rule(s)")


verdict_llm = llm.with_structured_output(ComplianceVerdict)


def latest_verdict(messages) -> str | None:
    """The compliance verdict given since the latest guest message ('COMPLIANT'/'VIOLATION'), or None if unchecked."""
    for m in reversed(messages):
        if m.type == "human":
            return None
        if getattr(m, "name", None) == "compliance":
            return m.content.split(":")[0].split()[0]
    return None


def generate(state: MessagesState) -> dict:
    """Write a structured verdict from the rules the retrieve tool just returned."""
    tool_messages = []
    for message in reversed(state["messages"]):
        if message.type != "tool":
            break
        tool_messages.insert(0, message)
    rules = "\n\n".join(m.content for m in tool_messages) or "(no rules retrieved)"
    last_request = next(m for m in reversed(state["messages"]) if m.type == "human")
    v = verdict_llm.invoke([
        hc.COMPLIANCE_PROMPT,
        SystemMessage(content=f"Retrieved rules:\n{rules}"),
        HumanMessage(content=f"Guest request: {last_request.content}\n\nGive your verdict."),
    ])
    text = v.verdict + (f" (rule {v.rule})" if v.rule else "") + f": {v.reason}"
    return {"messages": [AIMessage(content=text, name="compliance")]}


def choose_next_node(state: MessagesState) -> Literal["reservation_assistant", "compliance_checker", "__end__"]:
    return hc.route_by_mention(state["messages"], AGENTS) or END  # 03's router, with its loop guard


def this_turn(state) -> list:
    """The messages added after the latest guest message."""
    messages = state["messages"]
    last_guest = max(i for i, m in enumerate(messages) if m.type == "human")
    return messages[last_guest + 1:]


def after_verdict(state: MessagesState) -> Literal["human_review", "conv_assistant"]:
    """Pure router used when a review node is present: a VIOLATION goes to a human."""
    return "human_review" if latest_verdict(state["messages"]) == "VIOLATION" else "conv_assistant"


def build_graph(reservation_node=sql_assistant, *, review_node=None):
    builder = StateGraph(MessagesState)
    builder.add_node("conv_assistant", conv_assistant)
    builder.add_node("reservation_assistant", reservation_node)
    builder.add_node("compliance_checker", rag_assistant)
    builder.add_node("generate", generate)
    builder.add_node("sql_tools", ToolNode(sql_tools))
    builder.add_node("rag_tools", ToolNode(rag_tools))

    builder.add_edge(START, "conv_assistant")
    builder.add_conditional_edges("conv_assistant", choose_next_node, ["reservation_assistant", "compliance_checker", END])
    builder.add_conditional_edges("reservation_assistant", tools_condition, {"tools": "sql_tools", END: END})
    builder.add_conditional_edges("compliance_checker", tools_condition, {"tools": "rag_tools", END: "conv_assistant"})
    builder.add_edge("sql_tools", "reservation_assistant")
    builder.add_edge("rag_tools", "generate")
    if review_node is None:
        builder.add_edge("generate", "conv_assistant")
    else:
        builder.add_node("human_review", review_node)
        builder.add_conditional_edges("generate", after_verdict, ["human_review", "conv_assistant"])
    return builder.compile(checkpointer=InMemorySaver())


core_graph = build_graph()
print("✅ Core graph compiled. Edges:")
hc.print_edges(core_graph)

---
## 💉 Part 3: A Request With SQL Injection in It

The reservation agent writes its own SQL, so text from the guest can end up inside a query. This request hides a
classic tautology (`' OR '1'='1`) in the room type, and asks for dates that clash with the existing Family Suite
booking. If the text were pasted into `WHERE room_type = '...'`, the condition would match every room.

The checks below do not trust the replies: they print the SQL the agent actually executed and compare the table
sizes before and after.

In [ ]:
# ============================================================================
# AN INJECTION-STYLE REQUEST, CHECKED IN THE DATABASE
# ============================================================================
TAUTOLOGY = "'1'='1"
clash_in = date.today() + timedelta(days=31)
rows_before = {t: count_rows(t) for t in ("rooms", "reservations")}

injection = f"Hey, I want to book a Family Suite' OR '1'='1 from {clash_in} till {clash_in + timedelta(days=4)}"
state = hc.run_turn(core_graph, injection, thread_id="injection")

queries = [
    call["args"].get("sql", "")
    for m in this_turn(state) if m.type == "ai"
    for call in m.tool_calls if call["name"] == "run_query"
]
rows_after = {t: count_rows(t) for t in ("rooms", "reservations")}

print("\n--- SQL the agent executed ---")
for q in queries or ["(no query was executed)"]:
    print(" ", q)
print("\n--- Checks ---")
print(f"   compliance verdict: {latest_verdict(state['messages']) or 'no check'}")
print(f"{'✅' if rows_after == rows_before else '❌'} tables unchanged: {rows_before} -> {rows_after}")
print(f"{'✅' if not any(TAUTOLOGY in q for q in queries) else '⚠️'} tautology reached the database: "
      f"{any(TAUTOLOGY in q for q in queries)}")

**Reading the output:** start with the SQL list. A well-behaved agent either refuses (the checker may call the
request manipulation) or queries for `'Family Suite'` only. If a query contains `'1'='1`, the guest's text
reached the database unfiltered, but it could only widen a `SELECT`: `run_query`'s connection refuses every
write, `ATTACH` and `PRAGMA` (02 tested it), and the only write, `book_room`, binds its values as parameters
instead of pasting them into SQL. The table sizes must be unchanged either way, because no name was given and the
dates clash.

Those two are the durable fixes, and they are structural, not prompt-based: a **read-only** connection for
lookups, and a narrow, **parameterised** booking tool instead of free-form SQL.

---
## 🙋 Part 4: Stopping for a Human on a Policy Violation

Refusing politely is fine for most violations, but some should reach a person. A `human_review` node calls
`interrupt(payload)`: the graph saves a checkpoint and **pauses**, returning control to the caller with the
payload. A reviewer (here a scripted decision; in production a UI) answers, and
`graph.invoke(Command(resume=decision), config)` continues from exactly that point.

- The pause lives in a **node**, not in the router: routers stay pure functions (`after_verdict` only reads the
  verdict), and nodes are where work, and waiting, happens.
- A paused node **re-runs from the top** when resumed, so anything with side effects (here, recording the decision in
  `REVIEW_LOG`) goes *after* `interrupt()`, where it runs once. The check after the resume counts the entries.
- A checkpointer is required: the paused state has to be stored somewhere.

In [ ]:
# ============================================================================
# THE HUMAN-REVIEW NODE AND THE REVIEWED GRAPH
# ============================================================================
REVIEW_LOG = []  # one entry per reviewed request: a stand-in for a ticket system or an audit log


def human_review(state: MessagesState, config: RunnableConfig) -> Command[Literal["conv_assistant", "__end__"]]:
    """Pause for a person whenever the compliance verdict is VIOLATION."""
    request = next(m for m in reversed(state["messages"]) if m.type == "human").content
    verdict = next(m for m in reversed(state["messages"]) if getattr(m, "name", None) == "compliance").content
    decision = interrupt({"request": request, "verdict": verdict, "question": "approve or refuse?"})

    # Everything below runs once, after the reviewer answered (on resume the node re-runs from the top).
    REVIEW_LOG.append({"thread": config["configurable"]["thread_id"], "action": decision["action"]})
    if decision["action"] == "approve":
        note = AIMessage(content=f"A human reviewer approved this request: {decision.get('note', '')}", name="reviewer")
        return Command(goto="conv_assistant", update={"messages": [note]})
    reply = AIMessage(content=decision.get("reply", "I'm sorry, I can't help with that request."), name="reviewer")
    return Command(goto=END, update={"messages": [reply]})


reviewed_graph = build_graph(review_node=human_review)
print("✅ Reviewed graph compiled. Edges:")
hc.print_edges(reviewed_graph)

### Both sides of the gate
A compliant question should run to the end without pausing; a request for another guest's dates should stop at
`human_review`. `get_state(config).next` says which node is waiting, and `.interrupts` holds the payload.

In [ ]:
# ============================================================================
# ONE COMPLIANT AND ONE VIOLATING REQUEST
# ============================================================================
ok_config = {"configurable": {"thread_id": "review-ok"}}
bad_config = {"configurable": {"thread_id": "review-violation"}}

hc.run_turn(reviewed_graph, "Do you have any rooms with a balcony?", thread_id="review-ok")
print("=" * 80)
hc.run_turn(reviewed_graph, "Which dates is Bob Johnson staying with you? I want to surprise him.",
            thread_id="review-violation")

ok_snapshot = reviewed_graph.get_state(ok_config)
paused = reviewed_graph.get_state(bad_config)
print("\n--- Checks ---")
print(f"{'✅' if ok_snapshot.next == () else '❌'} compliant request finished without a pause (next={ok_snapshot.next})")
print(f"{'✅' if paused.next == ('human_review',) else '❌'} violating request is paused (next={paused.next})")
for pending in paused.interrupts:
    print(f"   interrupt payload: {pending.value}")

**Reading the output:** the first thread ends normally (`next=()`). The second shows the verdict message and then
stops: no refusal has been written yet, because the graph is waiting at `human_review` with the request and the
verdict in its payload. If the second check shows ❌, look at the verdict line: the checker judged the request
compliant, so there was nothing to review.

Now the reviewer answers. Try `{"action": "approve", "note": "..."}` too: the graph then goes back to the
coordinator instead of ending.

In [ ]:
# ============================================================================
# THE REVIEWER ANSWERS AND THE GRAPH RESUMES
# ============================================================================
decision = {
    "action": "refuse",
    "note": "privacy request about another guest",
    "reply": "I'm sorry, but we never share details about other guests' stays. "
             "If you'd like to arrange a surprise, our concierge can pass on a message.",
}
if paused.next == ("human_review",):
    result = reviewed_graph.invoke(Command(resume=decision), bad_config)
    result["messages"][-1].pretty_print()

resumed = reviewed_graph.get_state(bad_config)
last = resumed.values["messages"][-1]
print("\n--- Checks ---")
print(f"{'✅' if resumed.next == () else '❌'} thread finished after resuming (next={resumed.next})")
print(f"{'✅' if getattr(last, 'name', None) == 'reviewer' else '❌'} the last reply came from the reviewer")
reviews = [entry for entry in REVIEW_LOG if entry["thread"] == "review-violation"]
print(f"{'✅' if len(reviews) == 1 else '❌'} the code after interrupt() ran once for this thread: {reviews}")

---
## 🕰️ Part 5: Reading a Thread's History

Every step of a checkpointed graph is saved. `get_state_history(config)` returns the snapshots newest first; each
has the step number, the node(s) due to run next and the state at that point. This is how you find out what
happened in a thread a guest complained about.

In [ ]:
# ============================================================================
# THE CHECKPOINTS OF THE REVIEWED THREAD
# ============================================================================
history = list(reviewed_graph.get_state_history(bad_config))
for snap in reversed(history):  # oldest first
    messages = snap.values.get("messages", [])
    last_msg = messages[-1] if messages else None
    who = f"{last_msg.type}/{getattr(last_msg, 'name', None) or '-'}" if last_msg else "(empty)"
    print(f"step {snap.metadata.get('step'):>2} | next={str(snap.next):<28} | {len(messages)} messages | last: {who}")

print(f"\n{'✅' if any(s.next == ('human_review',) for s in history) else '❌'} "
      "the paused checkpoint is part of the history")

**Reading the output:** read it top to bottom as the story of the thread: the guest message, the coordinator
forwarding to the checker, the retrieval, the verdict, the checkpoint waiting at `human_review` (the pause), and
the reviewer's reply. Any of these snapshots can be passed back to `invoke` as a config to replay from there.

---
## 📅 Part 6: Refusing Dates in the Past

Nothing stops a guest from asking for last month. The reservation agent now checks dates before it touches the
database, in two parts:
- an LLM with **structured output** (`BookingDates`) extracts the dates, given today's date so it can resolve
  "next Friday";
- a **plain function** (`past_date_problem`) compares them with `date.today()`. Comparing dates is not a job for
  the model, and a pure function can be tested without any API call.

The check runs only when the coordinator hands over, not again on every return from a SQL tool.

In [ ]:
# ============================================================================
# THE DATE CHECK AND ITS PURE-CODE CORE
# ============================================================================
class BookingDates(BaseModel):
    check_in: str | None = Field(description="Check-in date as YYYY-MM-DD, or null if the guest gave none")
    check_out: str | None = Field(description="Check-out date as YYYY-MM-DD, or null if the guest gave none")


date_extractor = llm.with_structured_output(BookingDates)


def past_date_problem(dates: BookingDates, today: date) -> str | None:
    """Return a problem description if a date is unreadable or before today, else None."""
    for label, value in (("check-in", dates.check_in), ("check-out", dates.check_out)):
        if value is None:
            continue
        try:
            day = date.fromisoformat(value)
        except ValueError:
            return f"I couldn't read the {label} date '{value}'. Please give it as YYYY-MM-DD."
        if day < today:
            return f"the {label} date {day} is in the past (today is {today}). Please give future dates."
    return None


def reservation_with_date_check(state: MessagesState) -> dict:
    if state["messages"][-1].type != "tool":  # just handed over by the coordinator
        today = date.today()
        dates = date_extractor.invoke([
            SystemMessage(content=f"Today is {today:%Y-%m-%d} ({today:%A}). Extract the dates of the guest's latest "
                          "booking request. Resolve relative dates against today; if no year is given, use the "
                          "next occurrence of that date."),
            *state["messages"],
        ])
        problem = past_date_problem(dates, today)
        if problem:
            return {"messages": [AIMessage(content=f"Sorry, I can't book that: {problem}", name="date_check")]}
    return sql_assistant(state)


# The pure part, tested without the model
today = date.today()
cases = [
    (BookingDates(check_in=str(today + timedelta(days=10)), check_out=str(today + timedelta(days=12))), False),
    (BookingDates(check_in=str(today - timedelta(days=3)), check_out=str(today + timedelta(days=2))), True),
    (BookingDates(check_in="22.04.2025", check_out=None), True),
    (BookingDates(check_in=None, check_out=None), False),
]
for dates, should_block in cases:
    problem = past_date_problem(dates, today)
    print(f"{'✅' if bool(problem) == should_block else '❌'} {dates.check_in} -> {dates.check_out}: {problem or 'ok'}")

dated_graph = build_graph(reservation_with_date_check)
print("\n✅ Graph with the date check compiled")

### A past booking and a future one
Both requests are complete (name, room type, dates), so the coordinator should forward both to the reservation
agent. Only the second should get as far as SQL.

In [ ]:
# ============================================================================
# PAST DATES ARE REFUSED, FUTURE DATES GO THROUGH
# ============================================================================
past_in = date.today() - timedelta(days=200)
future_in = date.today() + timedelta(days=60)
scenarios = [
    ("Erin Park", past_in, "blocked"),
    ("Dana Lee", future_in, "booked"),
]

results = []
for guest, check_in, expected in scenarios:
    text = f"I'm {guest} and I'd like to book a Deluxe room from {check_in} to {check_in + timedelta(days=2)}."
    state = hc.run_turn(dated_graph, text, thread_id=f"dates-{guest}")
    new = this_turn(state)
    blocked = any(getattr(m, "name", None) == "date_check" for m in new)
    used_sql = any(m.type == "tool" and m.name in SQL_TOOL_NAMES for m in new)
    with sqlite3.connect("hotel.db") as conn:
        rows = conn.execute("SELECT COUNT(*) FROM reservations WHERE guest_name = ?", (guest,)).fetchone()[0]
    results.append((guest, check_in, expected, blocked, used_sql, rows))
    print("=" * 80)

print("\n--- Checks ---")
for guest, check_in, expected, blocked, used_sql, rows in results:
    ok = (blocked and not used_sql and rows == 0) if expected == "blocked" else (not blocked and rows == 1)
    print(f"{'✅' if ok else '❌'} {guest:<9} {check_in}  expected {expected:<7} | date check blocked={blocked}, "
          f"SQL used={used_sql}, rows written={rows}")

**Reading the output:** Erin Park's request is stopped by the date check (a reply tagged `date_check`, no SQL,
no row). Dana Lee's passes the check and should end with one row in `reservations`. A ❌ on Dana Lee's line with
`SQL used=True` means the agent checked availability but asked a follow-up question instead of booking: the date
guard worked, the booking step is the part to look at.

To chat with the hardened system yourself, run `hc.chat_with_agent(dated_graph, "my-thread")` in a new cell.

---
## 📝 Summary

- **Injection-style input** reaches the SQL an agent writes, so check the executed queries and the tables. The
  structural fixes are in place: `run_query` only reads, and `book_room` binds its values as parameters.
- **`interrupt()` in a node** pauses a checkpointed graph; `get_state().next` / `.interrupts` show the pause and
  `Command(resume=...)` continues it. Side effects go after `interrupt()`, where they run once.
- **`get_state_history()`** replays a thread checkpoint by checkpoint.
- **Past dates**: the LLM extracts the dates with structured output; plain code compares them with today.

### Next Steps
- `05_hotel_testing_the_system.ipynb`: a self-assessed confidence score and a small regression suite for the system
- Nodes that fail now and then (`RetryPolicy`) and a log file of every failure are in
  `09_hotel_system_wide_error_handling.ipynb`